# EEG Feature Extraction - Run Once, Use Everywhere

This notebook extracts EEG features from preprocessed EEG data and saves them to a pickle file.

**Parameterized**: Set `STRATEGY` to choose feature extraction approach.

**Run this notebook ONCE after EEG preprocessing completes.**

All other model notebooks will load the saved features instead of re-extracting.

---

## Feature Extraction Strategies

- **`regional`**: Regional averages only (16 features) - FASTEST, RECOMMENDED
- **`regional_with_channels`**: Regional + individual channels (96 features)
- **`non_temporal`**: Regional + temporal dynamics + lateralization (96 features)

In [1]:
# ============================================================================
# CONFIGURATION: Set extraction strategy
# ============================================================================
STRATEGY = 'regional_with_channels'  # Options: 'regional', 'regional_with_channels', 'non_temporal'
# ============================================================================

import sys
sys.path.append('../..')

import pickle
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from src.features.eeg_features import extract_eeg_features, get_feature_metadata

print(f"\n{'='*70}")
print(f"EEG FEATURE EXTRACTION: {STRATEGY.upper()} STRATEGY")
print(f"{'='*70}\n")
print(f"Feature extraction started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


EEG FEATURE EXTRACTION: REGIONAL_WITH_CHANNELS STRATEGY

Feature extraction started: 2026-03-28 15:30:30


## 1. Load Preprocessed EEG Data

Load the preprocessed EEG pickle file containing display_eeg data.

In [2]:
eeg_data_path = '../../data/eeg/Copy of preprocessed_eeg.pkl'

print(f"Loading EEG data from: {eeg_data_path}")
with open(eeg_data_path, 'rb') as f:
    eeg_df = pickle.load(f)

print(f"\n✓ Loaded {len(eeg_df)} trials")
print(f"  Unique subjects: {eeg_df['subject_date_id'].nunique()}")

# Check EEG data structure
sample_eeg = eeg_df['display_eeg'].iloc[0]
print(f"  EEG array shape: {sample_eeg.shape} (time × channels)")
print(f"\nColumns: {eeg_df.columns.tolist()}")

Loading EEG data from: ../../data/eeg/Copy of preprocessed_eeg.pkl

✓ Loaded 10855 trials
  Unique subjects: 85
  EEG array shape: (320, 20) (time × channels)

Columns: ['subject_date_id', 'trial_id', 'display_eeg']


## 2. Extract EEG Features

Extract features based on selected strategy:

### Regional (16 features)
- 4 frequency bands × 4 brain regions = 16 features
- **Features:** `eeg_{Delta,Theta,Alpha,Beta}_{Frontal,Central,Parietal,Occipital}`
- **Use case:** Baseline EEG features, fastest extraction

### Regional + Channels (96 features)
- 16 regional features + 80 individual channel features (4 bands × 20 channels)
- **Regional features:** `eeg_{band}_{region}` (16)
- **Channel features:** `eeg_{band}_{channel}` for each of:
  ```
  Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2
  ```
- **Use case:** Fine-grained spatial information per electrode

### Non-Temporal (96 features)
- 16 regional power + 48 temporal dynamics + 32 lateralization indices
- **Regional power:** `eeg_{band}_{region}` (16)
- **Temporal dynamics:** `eeg_{band}_{region}_{mean,std,slope}` (48)
- **Lateralization:** `eeg_{band}_{pair}_lateralization` (32) for pairs:
  ```
  Frontal_lateral (F7/F8), Frontal_medial (F3/F4), Frontal_pole (Fp1/Fp2),
  Central (C3/C4), Temporal_anterior (T3/T4), Temporal_posterior (T5/T6),
  Parietal (P3/P4), Occipital (O1/O2)
  ```
- **Use case:** Temporal evolution and hemispheric asymmetry

### Channel Regions (from chan_locs.sfp)
| Region | Channels |
|--------|----------|
| Frontal | Fp1, Fp2, F7, F3, Fz, F4, F8 |
| Central | T3, C3, Cz, C4, T4 |
| Parietal | T5, P3, Pz, P4, T6 |
| Occipital | O1, POz, O2 |

In [3]:
print(f"\nExtracting EEG features using '{STRATEGY}' strategy...\n")

eeg_features_df = extract_eeg_features(
    eeg_df=eeg_df,
    strategy=STRATEGY,
    fs=256,
    verbose=True
)

# Get feature columns
eeg_cols = [c for c in eeg_features_df.columns if c.startswith('eeg_')]

print(f"\n{'='*70}")
print(f"✓ Extracted {len(eeg_cols)} EEG features")
print(f"  Example features: {eeg_cols[:5]}")
if len(eeg_cols) > 5:
    print(f"  ... and {len(eeg_cols) - 5} more")
print(f"{'='*70}")


Extracting EEG features using 'regional_with_channels' strategy...

Extracting EEG features using 'regional_with_channels' strategy...
  Sampling rate: 256 Hz
  Trials: 10855
✓ Extracted 96 EEG features
  Regional power: 16 features
  Individual channels: 80 features

✓ Extracted 96 EEG features
  Example features: ['eeg_Delta_Frontal', 'eeg_Delta_Central', 'eeg_Delta_Parietal', 'eeg_Delta_Occipital', 'eeg_Theta_Frontal']
  ... and 91 more


## 3. Inspect Features

In [4]:
# Display sample data
print("\nSample EEG features:")
display(eeg_features_df.head(3))

# Feature statistics
print("\nFeature statistics:")
display(eeg_features_df[eeg_cols].describe())


Sample EEG features:


,subject_id,trial_id,eeg_Delta_Frontal,eeg_Delta_Central,eeg_Delta_Parietal,eeg_Delta_Occipital,eeg_Theta_Frontal,eeg_Theta_Central,eeg_Theta_Parietal,eeg_Theta_Occipital,...,eeg_Beta_Pz,eeg_Beta_F3,eeg_Beta_Fz,eeg_Beta_F4,eeg_Beta_C4,eeg_Beta_P4,eeg_Beta_POz,eeg_Beta_C3,eeg_Beta_Cz,eeg_Beta_O2
0,0831_1300_9M4VCHG,0_0831_1300_9M4VCHG,0.002910,0.002158,0.001471,0.000457,0.000441,0.000633,0.000268,0.000087,...,0.000305,0.000336,0.000200,0.000304,0.000817,0.000135,0.000127,0.000178,0.000353,0.000136
1,0831_1300_9M4VCHG,1_0831_1300_9M4VCHG,0.003351,0.001518,0.001177,0.000249,0.000478,0.000298,0.000277,0.000085,...,0.000411,0.000586,0.000425,0.000926,0.006093,0.000304,0.000201,0.001113,0.001009,0.001069
2,0831_1300_9M4VCHG,2_0831_1300_9M4VCHG,0.012950,0.005768,0.003191,0.005792,0.000977,0.000476,0.000425,0.000137,...,0.000295,0.000468,0.000259,0.000562,0.001137,0.000139,0.000120,0.000154,0.000567,0.000164



Feature statistics:


,eeg_Delta_Frontal,eeg_Delta_Central,eeg_Delta_Parietal,eeg_Delta_Occipital,eeg_Theta_Frontal,eeg_Theta_Central,eeg_Theta_Parietal,eeg_Theta_Occipital,eeg_Alpha_Frontal,eeg_Alpha_Central,...,eeg_Beta_Pz,eeg_Beta_F3,eeg_Beta_Fz,eeg_Beta_F4,eeg_Beta_C4,eeg_Beta_P4,eeg_Beta_POz,eeg_Beta_C3,eeg_Beta_Cz,eeg_Beta_O2
count,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000,...,10855.000000,10855.000000,10855.000000,10855.000000,1.085500e+04,10855.000000,10855.000000,10855.000000,10855.000000,10855.000000
mean,0.114168,0.101653,0.110994,0.122080,0.048825,0.044244,0.050229,0.057629,0.042691,0.041101,...,0.042833,0.058302,0.044876,0.051242,5.901658e-02,0.050128,0.048850,0.075365,0.038225,0.049891
std,0.482690,0.481650,0.528536,0.473655,0.093944,0.090880,0.097473,0.103581,0.085933,0.085105,...,0.101614,0.348176,0.154447,0.126735,2.753837e-01,0.130754,0.119830,0.557485,0.089944,0.103061
min,0.000103,0.000055,0.000049,0.000055,0.000036,0.000029,0.000018,0.000015,0.000024,0.000018,...,0.000010,0.000017,0.000005,0.000007,2.341966e-07,0.000008,0.000002,0.000006,0.000004,0.000009
25%,0.003387,0.002582,0.001998,0.002023,0.001005,0.000865,0.000707,0.000803,0.000551,0.000476,...,0.000451,0.000732,0.000480,0.000852,6.712422e-04,0.000394,0.000482,0.000548,0.000512,0.000513
50%,0.019589,0.014864,0.012142,0.012066,0.005378,0.004122,0.003931,0.004868,0.003598,0.003017,...,0.003427,0.005794,0.005450,0.007968,4.937927e-03,0.004235,0.004506,0.006343,0.003413,0.005802
75%,0.141660,0.127767,0.146060,0.145224,0.069529,0.066805,0.075480,0.079759,0.055543,0.053657,...,0.057803,0.060940,0.054101,0.058861,5.925480e-02,0.065126,0.067493,0.057682,0.053752,0.069008
max,36.801483,36.801483,36.801483,36.801483,2.192472,2.257807,2.974844,1.856315,1.765726,1.772360,...,4.563355,32.712884,12.388046,5.300057,1.403658e+01,7.541521,6.853849,44.743671,4.350238,2.905480


## 4. Categorize Features by Type

Group features by their type based on the extraction strategy.

In [5]:
feature_categories = {}

if STRATEGY == 'regional':
    feature_categories['regional_power'] = eeg_cols
    print(f"\nRegional power features: {len(feature_categories['regional_power'])}")

elif STRATEGY == 'regional_with_channels':
    from src.features.eeg_features import CHANNEL_REGIONS
    regional_cols = [c for c in eeg_cols if any(r in c for r in CHANNEL_REGIONS.keys())]
    channel_cols = [c for c in eeg_cols if c not in regional_cols]
    
    feature_categories['regional_power'] = regional_cols
    feature_categories['channel_power'] = channel_cols
    
    print(f"\nRegional power features: {len(regional_cols)}")
    print(f"Channel power features: {len(channel_cols)}")

elif STRATEGY == 'non_temporal':
    power_cols = [c for c in eeg_cols if not any(x in c for x in 
                  ['_mean', '_std', '_slope', '_lateralization'])]
    temporal_cols = [c for c in eeg_cols if any(x in c for x in 
                    ['_mean', '_std', '_slope'])]
    lat_cols = [c for c in eeg_cols if '_lateralization' in c]
    
    feature_categories['regional_power'] = power_cols
    feature_categories['temporal_dynamics'] = temporal_cols
    feature_categories['lateralization'] = lat_cols
    
    print(f"\nRegional power features: {len(power_cols)}")
    print(f"Temporal dynamics features: {len(temporal_cols)}")
    print(f"Lateralization features: {len(lat_cols)}")

print(f"\nTotal features: {len(eeg_cols)}")


Regional power features: 16
Channel power features: 80

Total features: 96


## 5. Prepare Metadata

Create comprehensive metadata for reproducibility.

In [6]:
metadata = get_feature_metadata(STRATEGY)
metadata.update({
    'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_trials': len(eeg_features_df),
    'n_subjects': eeg_features_df['subject_id'].nunique(),
    'input_file': eeg_data_path,
    'description': f'EEG features extracted using {STRATEGY} strategy'
})

print("\nMetadata:")
for key, value in metadata.items():
    if not isinstance(value, (dict, list)):
        print(f"  {key}: {value}")


Metadata:
  strategy: regional_with_channels
  sampling_rate: 256
  n_channels: 20
  n_regions: 4
  n_features: 96
  extraction_date: 2026-03-28 15:30:38
  n_trials: 10855
  n_subjects: 85
  input_file: ../../data/eeg/Copy of preprocessed_eeg.pkl
  description: EEG features extracted using regional_with_channels strategy


## 6. Save to Pickle File

Save features with metadata for use in other notebooks.

In [ ]:
# Output path based on strategy
output_dir = Path('../../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

if STRATEGY == 'regional':
    output_filename = 'eeg_features.pkl'
elif STRATEGY == 'regional_with_channels':
    output_filename = 'eeg_features_with_channels.pkl'
elif STRATEGY == 'non_temporal':
    output_filename = 'eeg_features_non_temporal.pkl'

output_path = output_dir / output_filename

# Prepare output data (matching main feature extraction format)
output_data = {
    'eeg_features_df': eeg_features_df,
    'feature_columns': eeg_cols,
    'feature_categories': feature_categories,
    'metadata': metadata
}

# Save
with open(output_path, 'wb') as f:
    pickle.dump(output_data, f)

file_size_mb = output_path.stat().st_size / 1024 / 1024

print(f"\n{'='*70}")
print(f"✓ EEG features saved to: {output_path}")
print(f"  File size: {file_size_mb:.2f} MB")
print(f"  Trials: {len(eeg_features_df)}")
print(f"  Subjects: {eeg_features_df['subject_id'].nunique()}")
print(f"  Features: {len(eeg_cols)}")
print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")

## 7. Verification

Verify the saved file can be loaded correctly.

In [8]:
# Test loading
print("\nVerifying saved file...")
with open(output_path, 'rb') as f:
    test_data = pickle.load(f)

print(f"✓ File loads successfully")
print(f"  Keys: {list(test_data.keys())}")
print(f"  Features: {len(test_data['feature_columns'])}")
print(f"  Trials: {len(test_data['eeg_features_df'])}")
print(f"  Strategy: {test_data['metadata']['strategy']}")
print("\n✓ Verification complete!")


Verifying saved file...
✓ File loads successfully
  Keys: ['eeg_features_df', 'feature_columns', 'feature_categories', 'metadata']
  Features: 96
  Trials: 10855
  Strategy: regional_with_channels

✓ Verification complete!


---

## Usage in Other Notebooks

To use these EEG features in fusion models or other analyses:

```python
import pickle

# Load EEG features
with open('../../data/features/eeg_features.pkl', 'rb') as f:
    eeg_data = pickle.load(f)

eeg_features_df = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
metadata = eeg_data['metadata']

# Merge with other modalities
merged_df = merged_df.merge(
    eeg_features_df,
    on=['subject_id', 'trial_id'],
    how='inner'
)
```

---

## Channel Configuration

All features use standardized channel configuration from `data/eeg/chan_locs.sfp`:

**20 EEG Channels (10-20 system):**
```
Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2
```

**4 Brain Regions:**
- **Frontal:** Fp1, Fp2, F7, F3, Fz, F4, F8
- **Central:** T3, C3, Cz, C4, T4
- **Parietal:** T5, P3, Pz, P4, T6
- **Occipital:** O1, POz, O2

**4 Frequency Bands:**
- Delta: 0.5-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz